# 8.11 — Attention mechanism

Attention turns a query into a weighted read over keys and values.

The lesson body is currently blank, so this notebook uses the plan's illustrative attention row. The core idea is still real: normalize query-key scores into a probability row, then use that row to mix value vectors.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

Attention computes normalized query-key scores and then reads values: $$\operatorname{Attention}(q,K,V)=\operatorname{softmax}(qK^\top)V$$. The lesson body is empty, so the numeric row here is illustrative as required by the plan.

In [ ]:

def attention(q, K, V, mask=None, temperature=1.0):
    scores = np.asarray(K) @ np.asarray(q) / temperature
    if mask is not None:
        scores = np.where(mask, scores, -1.0e9)
    weights = softmax(scores, axis=0)
    context = weights @ np.asarray(V)
    return context, weights, scores

q = np.asarray([1.0, 0.0])
K = np.asarray([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0]])
V = np.asarray([[10.0, 0.0], [0.0, 10.0], [5.0, 0.0]])
context, weights, scores = attention(q, K, V)
assert np.isclose(np.sum(weights), 1.0)
assert int(np.argmax(weights)) == 0
weights


A valid attention row is a probability distribution. Masking and temperature keep padded or overconfident links from becoming misleading alignments.

In [ ]:

masked_context, masked_weights, masked_scores = attention(q, K, V, mask=np.asarray([True, False, True]))
assert np.isclose(np.sum(masked_weights), 1.0)
assert masked_weights[1] < 1.0e-6
print("unmasked", weights)
print("masked", masked_weights)


## The dataset ladder

The inline ladder grows from one query-key-value toy to clean alignments, reordering, QA-like evidence, and diffuse distractors.

In [ ]:

rungs = build_attention_ladder()
for rung in rungs:
    print(rung["name"], "source length", len(rung["source"]), "targets", rung["targets"], "gold", rung["gold"])


## Run the same method across D1--D5

The same dot-product attention routine scores each target token against all source-token keys. Metric is alignment accuracy.

In [ ]:

def token_features(tokens):
    feats = []
    for token in tokens:
        feats.append([float(token == 0), float(token), math.sin(float(token)), math.cos(float(token))])
    return np.asarray(feats)


def attend_targets(source, targets, use_mask=True, temperature=1.0):
    K = token_features(source)
    V = K.copy()
    predictions = []
    rows = []
    mask = np.asarray([token != 0 for token in source]) if use_mask else None
    for target in targets:
        q_vec = token_features([target])[0]
        context, weights, scores = attention(q_vec, K, V, mask=mask, temperature=temperature)
        predictions.append(int(np.argmax(weights)))
        rows.append(weights)
    return predictions, np.asarray(rows)

results = []
for rung in rungs:
    pred, rows = attend_targets(rung["source"], rung["targets"])
    results.append({"rung": rung["name"], "length": len(rung["source"]), "accuracy": accuracy(pred, rung["gold"]), "rows": rows})

for row in results:
    print(f'{row["rung"]:28s} length={row["length"]:2d} alignment_accuracy={row["accuracy"]:.3f}')


## Results visualization

The panels are attention heatmaps. The curve summarizes alignment accuracy against sequence length and distractor complexity.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    pred, rows = attend_targets(rung["source"], rung["targets"])
    axes[0, index].imshow(rows, aspect="auto", cmap="Purples")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_xlabel("source")
    axes[0, index].set_ylabel("target")

axes[1, 0].plot([row["length"] for row in results], [row["accuracy"] for row in results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("source length")
axes[1, 0].set_ylabel("alignment accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: unnormalized or distractor-focused attention

Raw scores are not probabilities, and unmasked padding or diffuse distractors can steal attention. Row softmax, masking, and a checked temperature fix the behavior.

In [ ]:

d5 = rungs[-1]
wrong_pred, wrong_rows = attend_targets(d5["source"], d5["targets"], use_mask=False, temperature=0.25)
fixed_pred, fixed_rows = attend_targets(d5["source"], d5["targets"], use_mask=True, temperature=1.0)
raw_scores = token_features(d5["source"]) @ token_features([d5["targets"][0]])[0]
print("raw-score sum is not 1", float(np.sum(raw_scores)))
print("wrong unmasked D5 alignment accuracy", accuracy(wrong_pred, d5["gold"]))
print("fixed masked D5 alignment accuracy", accuracy(fixed_pred, d5["gold"]))
print("fixed first row sum", float(np.sum(fixed_rows[0])))


## Evaluate it + Practice

- Metric: alignment accuracy.
- No-skill baseline: always choose the first source token.
- Cheap sanity check: every attention row sums to 1.
- Ablation: remove masking or use tiny temperature on D5.
- Failure signals: attention rows peak on padding or have sums other than 1 before softmax.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Add a second matching source token and decide which one should win.

Practice 3: Change the temperature and describe sharper versus flatter rows.